In [1]:
import pandas as pd


# 1. Pandas로 원본 데이터 증식
repeat_factor = 5
df = pd.read_csv("restaurant-menus.csv")
df_expanded = pd.concat([df] * repeat_factor, ignore_index=True)
df_expanded["unique_row_id"] = df_expanded.index
expanded_file = f"restaurant-menus-x{repeat_factor}.csv"
df_expanded.to_csv(expanded_file, index=False)


In [1]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import broadcast, avg, regexp_extract, col
import time

# 안정화 대기
time.sleep(20)

# 1. SparkSession 생성
spark = SparkSession.builder \
    .appName("JoinBenchmark") \
    .master("spark://spark-master:7077") \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.autoBroadcastJoinThreshold", "-1") \
    .config("spark.hadoop.fs.defaultFS", "hdfs://namenode:9000") \
    .getOrCreate()


# 2. 자동 브로드캐스트 비활성화
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

# 3. 데이터 로드
menus_df = spark.read.csv("file:///home/jovyan/data/restaurant-menus-x5.csv", header=True, inferSchema=True)
restaurants_df = spark.read.csv("file:///home/jovyan/data/restaurants.csv", header=True, inferSchema=True)

# 4. price 전처리: 숫자만 추출해서 double로 변환
menus_df = menus_df.withColumn(
    "price_clean",
    regexp_extract(col("price"), r"([0-9]+(?:\.[0-9]+)?)", 1).cast("double")
)

# 5. Broadcast Join 함수
def run_broadcast_join(menus_df, restaurants_df):
    joined = menus_df.join(
        broadcast(restaurants_df),
        menus_df["restaurant_id"] == restaurants_df["id"],
        "inner"
    )
    return joined.groupBy(restaurants_df["name"]).agg(avg("price_clean").alias("avg_price"))

# 6. Shuffle Join 함수
def run_shuffle_join(menus_df, restaurants_df, num_partitions):
    menus_part = menus_df.repartition(num_partitions)
    restaurants_part = restaurants_df.repartition(num_partitions)
    joined = menus_part.join(
        restaurants_part,
        menus_part["restaurant_id"] == restaurants_part["id"],
        "inner"
    )
    return joined.groupBy(restaurants_part["name"]).agg(avg("price_clean").alias("avg_price"))

# 7. Broadcast Join 실행
start_bc = time.time()
result_bc = run_broadcast_join(menus_df, restaurants_df)
count_bc = result_bc.count()
end_bc = time.time()
print(f"[Broadcast Join] Time: {end_bc - start_bc:.2f} seconds, Result Count: {count_bc}")
result_bc.show(5)

# 8. Shuffle Join 테스트
for num_partitions in [4, 16, 64]:
    start_shuf = time.time()
    result_shuf = run_shuffle_join(menus_df, restaurants_df, num_partitions)
    count_shuf = result_shuf.count()
    end_shuf = time.time()
    print(f"[Shuffle Join - {num_partitions} partitions] Time: {end_shuf - start_shuf:.2f} seconds, Result Count: {count_shuf}")


[Broadcast Join] Time: 77.57 seconds, Result Count: 60707
+--------------------+------------------+
|                name|         avg_price|
+--------------------+------------------+
|Nelson Brothers C...| 4.532575757575757|
|    Ocean Restaurant|26.666666666666668|
|Captain D's (1284...| 7.332592592592597|
|  The Ice Cream Shop| 7.510509803921574|
| The Imperial Indian| 11.76217647058824|
+--------------------+------------------+
only showing top 5 rows

[Shuffle Join - 4 partitions] Time: 103.93 seconds, Result Count: 60707
[Shuffle Join - 16 partitions] Time: 159.91 seconds, Result Count: 60707
[Shuffle Join - 64 partitions] Time: 110.79 seconds, Result Count: 60707


In [3]:
# 상대경로 (현재 작업 디렉토리 기준)
result_bc.write.mode("overwrite").csv("hdfs://namenode:9000/user/jovyan/results/broadcast_join_csv")
result_shuf.write.mode("overwrite").csv("hdfs://namenode:9000/user/jovyan/results/shuffle_join_csv")

In [5]:
# 결과 Parquet 저장
result_bc.write.mode("overwrite").parquet("hdfs://namenode:9000/user/jovyan/results/broadcast_join_parquet")
result_shuf.write.mode("overwrite").parquet("hdfs://namenode:9000/user/jovyan/results/shuffle_join_parquet")


In [ ]:
import time
from pyspark.sql import SparkSession

spark = SparkSession.builder.getOrCreate()

# 경로 설정
csv_path = "hdfs:///user/jovyan/results/broadcast_join_csv"
parquet_path = "hdfs:///user/jovyan/results/broadcast_join_parquet"

# CSV 컬럼명 직접 지정
csv_columns = ["restaurant_name", "avg_price"]
parquet_columns = ["name", "avg_price"]  # Parquet 컬럼명 확인 결과 name으로 되어 있음

# CSV 데이터 읽기 (헤더 없음 가정)
df_csv = spark.read.option("header", False).option("inferSchema", True).csv(csv_path).toDF(*csv_columns)

# Parquet 데이터 읽기
df_parquet = spark.read.parquet(parquet_path)

# alias 설정
df_csv1 = df_csv.alias("csv1")
df_csv2 = df_csv.alias("csv2")

df_parquet1 = df_parquet.alias("parq1")
df_parquet2 = df_parquet.alias("parq2")

# CSV join key
csv_join_key = "restaurant_name"

# Parquet join key
parquet_join_key = "name"

# 1. CSV - Shuffle Join (브로드캐스트 OFF)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # 브로드캐스트 조인 OFF
start = time.time()
count_csv_shuffle = df_csv1.join(df_csv2, df_csv1[csv_join_key] == df_csv2[csv_join_key]).count()
print(f"[CSV - Shuffle Join] Join + count: {time.time() - start:.2f} sec, count={count_csv_shuffle}")

# 2. CSV - Broadcast Join (브로드캐스트 ON)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB
start = time.time()
count_csv_broadcast = df_csv1.join(df_csv2.hint("broadcast"), df_csv1[csv_join_key] == df_csv2[csv_join_key]).count()
print(f"[CSV - Broadcast Join] Join + count: {time.time() - start:.2f} sec, count={count_csv_broadcast}")


# 3. Parquet - Shuffle Join (브로드캐스트 OFF)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "-1")  # 브로드캐스트 조인 OFF
start = time.time()
count_parq_shuffle = df_parquet1.join(df_parquet2, df_parquet1[parquet_join_key] == df_parquet2[parquet_join_key]).count()
print(f"[Parquet - Shuffle Join] Join + count: {time.time() - start:.2f} sec, count={count_parq_shuffle}")

# 4. Parquet - Broadcast Join (브로드캐스트 ON)
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", "10485760")  # 10MB
start = time.time()
count_parq_broadcast = df_parquet1.join(df_parquet2.hint("broadcast"), df_parquet1[parquet_join_key] == df_parquet2[parquet_join_key]).count()
print(f"[Parquet - Broadcast Join] Join + count: {time.time() - start:.2f} sec, count={count_parq_broadcast}")


[CSV - Shuffle Join] Join + count: 0.50 sec, count=60843
[CSV - Broadcast Join] Join + count: 0.77 sec, count=60843
[Parquet - Shuffle Join] Join + count: 1.74 sec, count=60707
[Parquet - Broadcast Join] Join + count: 1.55 sec, count=60707
